# 07 – Economic evaluation and regimes
Long-only backtest of the walk-forward logistic regression signal at 0, 5, 10 and 20 bps, compared with persistence and buy-and-hold, plus the exploratory regime analysis.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, matthews_corrcoef

from google.colab import drive
drive.mount('/content/drive')

BASE     = "/content/drive/MyDrive/CS5500_dissertation"
FEATURES = f"{BASE}/features"
RESULTS  = f"{BASE}/results"
FIGURES  = f"{BASE}/figures"
os.makedirs(FIGURES, exist_ok=True)

TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]
SEED    = 42
DEV_END = "2023-12-31"

INITIAL_TRAIN, TEST_SIZE, STEP = 1000, 250, 250
TRADING_DAYS = 252

dev = {t: pd.read_csv(f"{FEATURES}/{t}_features.csv",
                      index_col=0, parse_dates=True).loc[:DEV_END]
       for t in TICKERS}

PREDICTORS = [c for c in dev["AAPL"].columns
              if not c.startswith("target") and c != "price_today"]

print(f"Loaded {len(TICKERS)} assets, {len(PREDICTORS)} predictors")

## 2. Generate dated predictions
The backtest needs predictions aligned to calendar dates, which the earlier notebooks did not retain.

In [ ]:
def expanding_walk_forward(n_rows, initial_train=INITIAL_TRAIN,
                           test_size=TEST_SIZE, step=STEP):
    start = initial_train
    while start + test_size <= n_rows:
        yield np.arange(0, start), np.arange(start, start + test_size)
        start += step


def walk_forward_predictions(df):
    """Sound protocol. Returns a dated frame of predictions and realised returns."""
    X = df[PREDICTORS].values
    y = df["target_direction"].values

    rows = []
    for tr, te in expanding_walk_forward(len(df)):
        scaler = StandardScaler().fit(X[tr])
        model  = LogisticRegression(max_iter=1000, random_state=SEED)
        model.fit(scaler.transform(X[tr]), y[tr])

        pred = model.predict(scaler.transform(X[te]))

        rows.append(pd.DataFrame({
            "pred_model":  pred,
            "actual_dir":  y[te],
            "next_return": df["target_return"].values[te],
            "pred_persist": (df["ret_lag_1"].values[te] > 0).astype(int),
        }, index=df.index[te]))

    return pd.concat(rows)


predictions = {}
for t in TICKERS:
    predictions[t] = walk_forward_predictions(dev[t])
    p = predictions[t]
    print(f"{t}: {len(p)} predictions, {p.index.min().date()} to {p.index.max().date()}, "
          f"accuracy {accuracy_score(p.actual_dir, p.pred_model):.4f}")

## 3. The backtest
Long when the model predicts up, flat otherwise; costs charged on every position change.

In [ ]:
def backtest(signal, next_return, cost_bps):
    """Long when signal is 1, flat when 0. Cost charged on position changes.

    Returns a dict of performance statistics.
    """
    signal      = np.asarray(signal, dtype=float)
    next_return = np.asarray(next_return, dtype=float)

    # position changes: compare to previous day, starting flat
    prev = np.concatenate([[0.0], signal[:-1]])
    trades = np.abs(signal - prev)

    cost = trades * (cost_bps / 10_000)
    strategy_return = signal * next_return - cost

    equity = np.cumprod(1 + strategy_return)
    peak   = np.maximum.accumulate(equity)
    drawdown = equity / peak - 1

    ann_return = strategy_return.mean() * TRADING_DAYS
    ann_vol    = strategy_return.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe     = ann_return / ann_vol if ann_vol > 0 else np.nan

    return {
        "total_return": equity[-1] - 1,
        "ann_return":   ann_return,
        "ann_vol":      ann_vol,
        "sharpe":       sharpe,
        "max_drawdown": drawdown.min(),
        "n_trades":     int(trades.sum()),
        "turnover_pa":  trades.sum() / len(signal) * TRADING_DAYS,
        "equity":       equity,
    }


print("Backtest function defined")

In [ ]:
COSTS = [0, 5, 10, 20]

records = []

for t in TICKERS:
    p = predictions[t]

    strategies = {
        "model":       p["pred_model"].values,
        "persistence": p["pred_persist"].values,
        "buy_hold":    np.ones(len(p)),
    }

    for name, signal in strategies.items():
        for c in COSTS:
            r = backtest(signal, p["next_return"].values, c)
            records.append({
                "ticker": t, "strategy": name, "cost_bps": c,
                "total_return": round(r["total_return"], 4),
                "ann_return":   round(r["ann_return"], 4),
                "sharpe":       round(r["sharpe"], 4),
                "max_drawdown": round(r["max_drawdown"], 4),
                "turnover_pa":  round(r["turnover_pa"], 1),
            })

backtest_results = pd.DataFrame(records)

print("Sharpe ratio by strategy and cost level\n")
print(backtest_results.pivot_table(index=["ticker", "strategy"],
                                   columns="cost_bps",
                                   values="sharpe").round(3).to_string())

backtest_results.to_csv(f"{RESULTS}/backtest.csv", index=False)

In [ ]:
print("Mean across the five assets\n")

summary = (backtest_results.groupby(["strategy", "cost_bps"])
                           [["ann_return", "sharpe", "max_drawdown", "turnover_pa"]]
                           .mean().round(4))
print(summary.to_string())

print("\n\nThe cost of trading\n")
for s in ["model", "persistence", "buy_hold"]:
    free = summary.loc[(s, 0), "sharpe"]
    paid = summary.loc[(s, 10), "sharpe"]
    turn = summary.loc[(s, 0), "turnover_pa"]
    print(f"  {s:<12} turnover {turn:>6.1f} trades/yr   "
          f"Sharpe {free:+.3f} -> {paid:+.3f} at 10bps")

print("\nBuy and hold trades once and pays the cost once. A daily-signal")
print("strategy pays it repeatedly, which is why turnover matters as much")
print("as accuracy.")

## 4. Does the model beat buy-and-hold?
Mean Sharpe ratio of the model against buy-and-hold at each cost level.

In [ ]:
comparison = []

for c in COSTS:
    sub = backtest_results.query("cost_bps == @c")
    m = sub.query("strategy == 'model'").set_index("ticker")["sharpe"]
    b = sub.query("strategy == 'buy_hold'").set_index("ticker")["sharpe"]

    comparison.append({
        "cost_bps":        c,
        "model_sharpe":    round(m.mean(), 4),
        "buyhold_sharpe":  round(b.mean(), 4),
        "difference":      round(m.mean() - b.mean(), 4),
        "assets_model_wins": int((m > b).sum()),
    })

comp = pd.DataFrame(comparison)
print("Model versus buy-and-hold, mean Sharpe across five assets\n")
print(comp.to_string(index=False))

comp.to_csv(f"{RESULTS}/model_vs_buyhold.csv", index=False)

## 5. Regime analysis
Three periods with materially different market conditions.

In [ ]:
REGIMES = {
    "pre_covid":  ("2009-03-05", "2020-01-31"),
    "covid":      ("2020-02-01", "2021-12-31"),
    "post_covid": ("2022-01-01", "2023-01-27"),
}

regime_rows = []

for t in TICKERS:
    p = predictions[t]

    for name, (start, end) in REGIMES.items():
        sub = p.loc[start:end]
        if len(sub) < 30:
            continue

        bt = backtest(sub["pred_model"].values, sub["next_return"].values, 10)

        regime_rows.append({
            "ticker": t, "regime": name, "n": len(sub),
            "accuracy": round(accuracy_score(sub.actual_dir, sub.pred_model), 4),
            "mcc":      round(matthews_corrcoef(sub.actual_dir, sub.pred_model), 4),
            "pct_up":   round(sub.actual_dir.mean(), 4),
            "sharpe_10bps": round(bt["sharpe"], 4),
            "realised_vol": round(sub.next_return.std() * np.sqrt(TRADING_DAYS), 4),
        })

regimes = pd.DataFrame(regime_rows)

print("Performance by market regime, mean across assets\n")
print(regimes.groupby("regime")[["n", "accuracy", "mcc", "pct_up",
                                 "sharpe_10bps", "realised_vol"]]
              .mean().round(4).to_string())

print("\n\nPer asset\n")
print(regimes.pivot(index="ticker", columns="regime",
                    values="accuracy").round(4).to_string())

regimes.to_csv(f"{RESULTS}/regime_analysis.csv", index=False)

## 6. Equity curve figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

p = predictions["SPY"]
dates = p.index

for name, signal, style in [
    ("model",       p["pred_model"].values,  "-"),
    ("persistence", p["pred_persist"].values, "--"),
    ("buy and hold", np.ones(len(p)),        ":"),
]:
    bt = backtest(signal, p["next_return"].values, 10)
    axes[0].plot(dates, bt["equity"], style, label=f"{name} (SR {bt['sharpe']:.2f})")

axes[0].set_title("SPY equity curves, 10 bps costs")
axes[0].set_ylabel("cumulative growth of 1")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45)

# Sharpe decay with cost
for strategy in ["model", "persistence", "buy_hold"]:
    s = [backtest_results.query("strategy == @strategy and cost_bps == @c")["sharpe"].mean()
         for c in COSTS]
    axes[1].plot(COSTS, s, marker="o", label=strategy)

axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("transaction cost (basis points)")
axes[1].set_ylabel("mean Sharpe ratio")
axes[1].set_title("Sharpe decay with trading costs")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{FIGURES}/economic_evaluation.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {FIGURES}/economic_evaluation.png")

## 7. Append to the results log

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append(f"## {today} - Notebook 07: economic evaluation and regimes")
lines.append("")
lines.append("**Backtest design**")
lines.append("- Long when the model predicts up, flat otherwise (long-only)")
lines.append("- Cost charged on every position change, at 0/5/10/20 bps")
lines.append("- Predictions from logistic regression under the sound protocol")
lines.append("- Omits slippage, market impact and borrowing costs, so results "
             "are an optimistic bound")
lines.append("")
lines.append("**Performance, mean across five assets**")
lines.append("")
lines.append(summary.to_string())
lines.append("")
lines.append("**Model versus buy-and-hold**")
lines.append("")
lines.append(comp.to_string(index=False))
lines.append("")
lines.append("**Regime analysis (exploratory; boundaries chosen with hindsight)**")
lines.append("")
lines.append(regimes.groupby("regime")[["n","accuracy","mcc","pct_up",
                                        "sharpe_10bps","realised_vol"]]
                    .mean().round(4).to_string())
lines.append("")
lines.append("**Figure:** figures/economic_evaluation.png")
lines.append("")
lines.append("**Next:** final holdout evaluation on 2024-2025, reported once")

entry = "\n".join(lines)

with open(f"{BASE}/results_log.md", "a") as f:
    f.write(entry)

print(entry)